In [1]:
# ==== 0) ENV + DEP CHECK (safe to re-run) ====
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"  # avoid torchvision ops (NMS)

In [2]:
# If you want to pin versions right here too (optional):
# %pip install --quiet --no-cache-dir "transformers==4.44.2" "accelerate==0.34.0" "peft==0.12.0" "datasets>=2.20.0" sentencepiece

In [3]:
import torch, json
from pathlib import Path

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [4]:
# ==== 1) MAKE A TOY DATASET (Alpaca-ish JSONL) ====
data_dir = Path("/workspace/data/processed")
data_dir.mkdir(parents=True, exist_ok=True)
jsonl_path = data_dir / "train.jsonl"

rows = [
    {"instruction":"Translate English to French", "input":"Hello",   "output":"Bonjour"},
    {"instruction":"Translate English to French", "input":"Goodbye", "output":"Au revoir"},
    {"instruction":"What is 2+2?",               "input":"",        "output":"4"},
    {"instruction":"Who wrote Hamlet?",          "input":"",        "output":"William Shakespeare"},
]
with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("✅ Saved dataset:", jsonl_path)

✅ Saved dataset: /workspace/data/processed/train.jsonl


In [5]:
# ==== 2) LOAD + FORMAT ====
from datasets import load_dataset

ds = load_dataset("json", data_files=str(jsonl_path), split="train")

def to_prompt(ex):
    instr = ex.get("instruction","")
    inp   = ex.get("input","")
    out   = ex.get("output","")
    prompt = f"### Instruction:\n{instr}\n\n### Input:\n{inp}\n\n### Response:\n"
    return {"prompt": prompt, "text": prompt + out, "labels": out}

ds = ds.map(to_prompt)
print(ds[0])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

{'instruction': 'Translate English to French', 'input': 'Hello', 'output': 'Bonjour', 'prompt': '### Instruction:\nTranslate English to French\n\n### Input:\nHello\n\n### Response:\n', 'text': '### Instruction:\nTranslate English to French\n\n### Input:\nHello\n\n### Response:\nBonjour', 'labels': 'Bonjour'}


In [6]:
# ==== 3) TOKENIZE (mask prompt so loss is on the response only) ====
from transformers import AutoTokenizer

BASE = "sshleifer/tiny-gpt2"  # super small for demo; swap later to a real base
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

MAX_LEN = 256
def tok_fn(ex):
    enc = tok(ex["text"], truncation=True, max_length=MAX_LEN)
    # create labels that ignore (=-100) the prompt tokens
    prompt_ids = tok(ex["prompt"], truncation=True, max_length=MAX_LEN)["input_ids"]
    labels = enc["input_ids"][:]
    m = min(len(prompt_ids), len(labels))
    labels[:m] = [-100] * m
    enc["labels"] = labels
    return enc

cols_to_remove = [c for c in ds.column_names if c not in ("text","prompt","labels")]
ds_tok = ds.map(tok_fn, remove_columns=cols_to_remove)
print("Tokenized example keys:", ds_tok.features)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenized example keys: {'prompt': Value('string'), 'text': Value('string'), 'labels': List(Value('int64')), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


In [7]:
# ==== 4) MODEL + LoRA (PEFT) ====
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(BASE).to(device)

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn","c_proj"],  # GPT-2 attention/proj layers
)
model = get_peft_model(model, lora_cfg)

pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/peft/tuners/lora/layer.py:1091: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [8]:
# ==== 5) TRAIN (very short demo run) ====
out_dir = Path("/workspace/adapters/myrun-demo")
out_dir.mkdir(parents=True, exist_ok=True)

args = TrainingArguments(
    output_dir=str(out_dir),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=1,   # increase later
    logging_steps=1,
    save_steps=50,
    fp16=torch.cuda.is_available(),
)

dcoll = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)
trainer = Trainer(model=model, args=args, train_dataset=ds_tok, data_collator=dcoll)
trainer.train()

/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss
1,10.823300
2,10.831200


TrainOutput(global_step=2, training_loss=10.82724666595459, metrics={'train_runtime': 1.1972, 'train_samples_per_second': 3.341, 'train_steps_per_second': 1.671, 'total_flos': 290304.0, 'train_loss': 10.82724666595459, 'epoch': 1.0})

In [9]:
# ==== 6) SAVE ADAPTER + TOKENIZER ====
trainer.model.save_pretrained(str(out_dir))
tok.save_pretrained(str(out_dir))
print("✅ LoRA adapter saved to:", out_dir)

✅ LoRA adapter saved to: /workspace/adapters/myrun-demo


In [10]:
# ==== 7) QUICK GENERATION TEST (no pipeline) ====
from transformers import GenerationConfig

test_prompt = "### Instruction:\nTranslate English to French\n\n### Input:\nHello\n\n### Response:\n"
inputs = tok(test_prompt, return_tensors="pt").to(device)
gen_cfg = GenerationConfig(max_new_tokens=20, do_sample=False)
with torch.no_grad():
    gen = model.generate(**inputs, generation_config=gen_cfg)
print("\n--- Generated ---\n", tok.decode(gen[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- Generated ---
 ### Instruction:
Translate English to French

### Input:
Hello

### Response:
 factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors
